# WESAD: Dataset Selection and Final Project Planning

**Jillian Melbourne** · HSE.751, Fall 2026 · Week 3 Lab

**Dataset:** WESAD (Wearable Stress and Affect Detection), UC Irvine Machine Learning Repository
<https://archive.ics.uci.edu/dataset/465/wesad+wearable+stress+and+affect+detection>

**Proposed task:** binary classification of *stress* vs. *non-stress* from 60-second windows of chest and wrist physiological signals.

### Contents
1. Data selection and justification
2. Data import and raw structure
3. From raw signals to a feature table (windowing and feature engineering)
4. Descriptive statistics
5. Exploratory visualizations
6. Data quality assessment
7. Project planning
8. Requirements check and session info

**How to run:** *Runtime → Run all*. The notebook downloads the dataset (about 2.5 GB) from UCI on first run, extracts only the 15 subject `.pkl` files, and builds the feature table. A full run on a standard Colab CPU runtime takes roughly 10–15 minutes, most of it the download. No GPU is needed.

In [ ]:
# Setup: imports, versions, and global settings
import sys, os, re, gc, glob, time, pickle, zipfile, platform, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
from scipy import signal
import matplotlib
import matplotlib.pyplot as plt

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
IN_COLAB = "google.colab" in sys.modules

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "font.size": 10})

FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
OUT_DIR = Path("data/processed"); OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python      {platform.python_version()}  ({platform.platform()})")
for name, mod in [("numpy", np), ("pandas", pd), ("scipy", scipy), ("matplotlib", matplotlib)]:
    print(f"{name:<11} {mod.__version__}")
print(f"Running in Colab: {IN_COLAB}")

## 1. Data selection and justification

**Dataset:** WESAD (Wearable Stress and Affect Detection), Schmidt et al., 2018.
**Source:** UC Irvine Machine Learning Repository, <https://archive.ics.uci.edu/dataset/465/wesad+wearable+stress+and+affect+detection>

**Why I selected this dataset.** My background is in signal processing for deep brain stimulation and wearables, so this dataset feels like familiar territory with new tools. WESAD includes physiological and motion signals from 15 participants wearing a chest sensor (RespiBAN) and a wrist sensor (Empatica E4) during a lab stress protocol: ECG, blood volume pulse, electrodermal activity, respiration, skin temperature, EMG, and acceleration. I'm excited to spend more time on feature engineering and then applying ML techniques to those features.

**Clinical motivation.** Stress is largely invisible and hard to measure beyond self-report, yet it has profound health effects, from cardiovascular disease to mental health. Quantifying stress objectively is an interesting clinical question, and wearables offer one of the most promising routes to doing it passively and continuously.

**Fit with my goals.** I'm interested in applying ML and AI methods to physiological and wearable data. This project is a direct way to build that skill set: a complete pipeline from raw signal processing through model evaluation, with a full write-up for my GitHub.

**The study protocol.** Each participant went through a baseline period (reading magazines, about 20 min), an amusement condition (funny video clips, about 6.5 min), a stress condition (the Trier Social Stress Test: public speaking plus mental arithmetic in front of a panel, about 10 min), and guided meditation periods. Every sample of the chest recording carries a condition label.

### Does WESAD meet the project criteria?

| Requirement | How WESAD meets it |
|---|---|
| Binary classification | Labels are recoded to **stress (1)** vs. **non-stress (0)** = baseline + amusement. This is the binary task defined by the dataset's authors. |
| ≥ 300 observations | The raw data are continuous signals, so the unit of observation is a 60-second window. Windowing yields roughly 1,000 labeled windows (exact count verified in Section 8). |
| ≥ 10 predictor variables | Section 3 engineers 32 physiological features per window (18 chest, 14 wrist). |

One honest caveat: WESAD does not arrive as a ready-made table. Turning signals into observations is part of the project, and it means the ~1,000 windows come from only **15 people**. That shapes how the model must be evaluated (Section 7).

**Backup dataset.** If aligning the two devices' data streams turns into more of a rabbit hole than this course permits, my fallback is the UCI Mushroom dataset (<https://archive.ics.uci.edu/dataset/73/mushroom>): 8,124 mushrooms described by 22 categorical features, labeled edible or poisonous. In terms of health impact, avoiding a toxic mushroom may be the most direct form of preventive health there is.

## 2. Data import and raw structure

### 2.1 Download and extract
The UCI archive holds one folder per subject (`S2` … `S17`; S1 and S12 were dropped by the authors because of sensor malfunction). Each folder contains raw device exports, a questionnaire file, and a synchronized Python pickle `S#.pkl` with both devices' signals and the condition labels. This notebook uses only the pickles.

*Optional:* set `USE_DRIVE_CACHE = True` to keep the extracted files in Google Drive so later runs skip the download.

In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/465/wesad+wearable+stress+and+affect+detection.zip"
USE_DRIVE_CACHE = False   # True = store the extracted pickles in Google Drive (Colab only)

if IN_COLAB and USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_DIR = "/content/drive/MyDrive/WESAD_data"
elif IN_COLAB:
    DEFAULT_DIR = "/content/WESAD_data"
else:
    DEFAULT_DIR = "data/raw"
DATA_DIR = Path(os.environ.get("WESAD_DATA_DIR", DEFAULT_DIR))
DATA_DIR.mkdir(parents=True, exist_ok=True)

SUBJECT_FILE = re.compile(r"^S(\d+)\.pkl$")

def find_subject_files(root):
    # Return the per-subject pickles under root, sorted by subject number
    files = [p for p in Path(root).rglob("S*.pkl") if SUBJECT_FILE.match(p.name)]
    return sorted(files, key=lambda p: int(SUBJECT_FILE.match(p.name).group(1)))

def extract_pickles(zip_path, dest):
    # Extract only the subject pickles; handles a zip nested inside the UCI zip
    with zipfile.ZipFile(zip_path) as zf:
        members = zf.namelist()
        pkl = [m for m in members if SUBJECT_FILE.match(Path(m).name)]
        if pkl:
            for m in pkl:
                print("  extracting", m)
                zf.extract(m, dest)
            return
        inner = [m for m in members if m.lower().endswith(".zip") and "wesad" in m.lower()]
        if not inner:
            raise RuntimeError(f"No subject pickles or inner WESAD zip found in {zip_path}")
        print("  found nested archive:", inner[0])
        inner_path = Path(zf.extract(inner[0], dest))
    extract_pickles(inner_path, dest)
    inner_path.unlink()

subject_files = find_subject_files(DATA_DIR)
if not subject_files:
    zip_path = DATA_DIR / "wesad_uci.zip"
    if not zip_path.exists():
        print(f"Downloading WESAD (~2.5 GB) from UCI to {zip_path} ...")
        t0 = time.time()
        def _progress(blocks, block_size, total):
            done = blocks * block_size
            if total > 0 and blocks % 2000 == 0:
                print(f"  {done/1e9:5.2f} / {total/1e9:.2f} GB  ({time.time()-t0:4.0f} s)")
        urllib.request.urlretrieve(DATA_URL, zip_path, _progress)
        print(f"Download finished in {time.time()-t0:.0f} s")
    print("Extracting subject pickles ...")
    extract_pickles(zip_path, DATA_DIR)
    zip_path.unlink()   # free ~2.5 GB of disk
    subject_files = find_subject_files(DATA_DIR)

print(f"\n{len(subject_files)} subject files found:")
print(", ".join(p.stem for p in subject_files))

### 2.2 Raw structure of one subject
Each pickle is a nested dictionary. It was written with Python 2, so it is loaded with `encoding="latin1"`.

In [ ]:
def load_subject(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="latin1")

example = load_subject(subject_files[0])
print("Top-level keys:", list(example.keys()))
print("Subject ID:", example["subject"])

FS = {"chest": {k: 700 for k in ["ACC", "ECG", "EMG", "EDA", "Temp", "Resp"]},
      "wrist": {"ACC": 32, "BVP": 64, "EDA": 4, "TEMP": 4},
      "label": 700}
UNITS = {("chest", "ACC"): "g", ("chest", "ECG"): "mV", ("chest", "EMG"): "mV", ("chest", "EDA"): "µS",
         ("chest", "Temp"): "°C", ("chest", "Resp"): "%", ("wrist", "ACC"): "1/64 g",
         ("wrist", "BVP"): "a.u.", ("wrist", "EDA"): "µS", ("wrist", "TEMP"): "°C"}

rows = []
for device, chans in example["signal"].items():
    for ch, arr in chans.items():
        arr = np.asarray(arr)
        fs = FS[device][ch]
        rows.append({"device": device, "channel": ch, "shape": arr.shape, "dtype": str(arr.dtype),
                     "fs_Hz": fs, "duration_min": len(arr) / fs / 60, "units": UNITS[(device, ch)],
                     "min": np.nanmin(arr), "max": np.nanmax(arr), "n_NaN": int(np.isnan(arr).sum())})
lab = np.asarray(example["label"])
rows.append({"device": "label", "channel": "label", "shape": lab.shape, "dtype": str(lab.dtype),
             "fs_Hz": 700, "duration_min": len(lab) / 700 / 60, "units": "code",
             "min": lab.min(), "max": lab.max(), "n_NaN": 0})
structure = pd.DataFrame(rows)
display(structure)

**Label coding (from the WESAD readme):** 0 = not defined / transient, 1 = baseline, 2 = stress, 3 = amusement, 4 = meditation, 5–7 = "should be ignored". Labels are sampled at 700 Hz, aligned with the chest signals.

**Target variable.** Only samples labeled 1, 2, or 3 are used. The binary target is `stress = 1` for label 2 and `stress = 0` for labels 1 and 3 (baseline + amusement). Meditation (4) is excluded to match the authors' binary task; it could be added to the non-stress class later as a sensitivity analysis.

In [ ]:
LABEL_NAMES = {0: "transient/undefined", 1: "baseline", 2: "stress", 3: "amusement",
               4: "meditation", 5: "ignore (5)", 6: "ignore (6)", 7: "ignore (7)"}
vc = pd.Series(lab).value_counts().sort_index()
label_table = pd.DataFrame({"meaning": [LABEL_NAMES.get(int(k), "?") for k in vc.index],
                            "n_samples": vc.values, "minutes": vc.values / 700 / 60,
                            "share_%": 100 * vc.values / vc.sum()}, index=vc.index.rename("label"))
print(f"Label distribution for {example['subject']} (raw 700 Hz samples):")
display(label_table)

### Visualization 1: raw signals across the protocol (one subject)
Shaded bands show the study condition. This is the clearest way to see what the classifier will be asked to separate, and whether the stress response is visible by eye.

In [ ]:
COND_COLORS = {1: "#9db4c0", 2: "#e07a5f", 3: "#81b29a", 4: "#c9b6e4"}

def lowpass_decimate(x, fs, target_fs, cutoff):
    sos = signal.butter(4, cutoff, btype="low", fs=fs, output="sos")
    step = int(round(fs / target_fs))
    return signal.sosfiltfilt(sos, np.asarray(x, dtype=float).ravel())[::step]

ch, wr = example["signal"]["chest"], example["signal"]["wrist"]
lab4 = lab[::175]                                    # labels at 4 Hz
t4 = np.arange(len(lab4)) / 4 / 60                   # minutes
panels = [
    ("Chest EDA (µS)", lowpass_decimate(ch["EDA"], 700, 4, 1.0), 4),
    ("Wrist EDA (µS)", np.asarray(wr["EDA"]).ravel(), 4),
    ("Chest respiration (%)", lowpass_decimate(ch["Resp"], 700, 4, 1.0), 4),
    ("Wrist skin temp (°C)", np.asarray(wr["TEMP"]).ravel(), 4),
    ("Wrist ACC magnitude (g)", np.linalg.norm(np.asarray(wr["ACC"]) / 64, axis=1), 32),
]
fig, axes = plt.subplots(len(panels), 1, figsize=(12, 9), sharex=True)
for ax, (title, y, fs) in zip(axes, panels):
    ax.plot(np.arange(len(y)) / fs / 60, y, lw=0.6, color="#333333")
    ax.set_ylabel(title, rotation=0, ha="right", va="center", fontsize=9)
    # shade conditions
    change = np.flatnonzero(np.diff(lab4) != 0) + 1
    bounds = np.r_[0, change, len(lab4)]
    for a, b in zip(bounds[:-1], bounds[1:]):
        c = int(lab4[a])
        if c in COND_COLORS:
            ax.axvspan(t4[a], t4[b - 1], color=COND_COLORS[c], alpha=0.35, lw=0)
handles = [plt.Rectangle((0, 0), 1, 1, color=COND_COLORS[k], alpha=0.5) for k in COND_COLORS]
axes[0].legend(handles, [LABEL_NAMES[k] for k in COND_COLORS], ncol=4, loc="upper left",
               bbox_to_anchor=(0, 1.45), frameon=False)
axes[-1].set_xlabel("Time since start of recording (min)")
fig.suptitle(f"Subject {example['subject']}: raw chest and wrist signals by study condition", y=1.0)
fig.tight_layout()
fig.savefig(FIG_DIR / "fig1_raw_signals_by_condition.png", bbox_inches="tight")
plt.show()

What to look for: electrodermal activity is expected to climb during the stress condition (a sympathetic nervous system response), and respiration to become faster and less regular. Wrist signals are typically noisier than chest signals, and bursts of wrist motion tend to appear between conditions, when participants moved between rooms or filled in questionnaires. The per-subject versions of these patterns are quantified in Sections 4–5.

## 3. From raw signals to a feature table

The two devices record at very different rates (chest: 700 Hz for everything; wrist: 4 to 64 Hz). Rather than resampling all signals onto one grid, I align them **in time**: every window is defined in seconds, and each signal is sliced at its own sampling rate. The pickles are already synchronized by the dataset authors, so the same second on both devices refers to the same moment.

**Windowing.** 60-second windows with a 30-second step (50% overlap). A window is kept only if *every* label sample in it is the same condition (1, 2, or 3). Windows that straddle a condition change or contain transient/ignored labels are dropped and counted in Section 6.

**Features per window (32 total).**

| Signal | Preprocessing | Features |
|---|---|---|
| Chest ECG (700 Hz) | 5–15 Hz band-pass, derivative-squared energy, R-peak detection | HR mean, HR SD, SDNN, RMSSD |
| Chest EDA (700 Hz) | 1 Hz low-pass, decimated to 4 Hz | mean, SD, min, max, slope, # SCR peaks |
| Chest respiration (700 Hz) | decimated to 10 Hz, 0.1–0.7 Hz band-pass, breath detection | breathing rate, amplitude SD |
| Chest temperature | 1 Hz low-pass, 4 Hz | mean, SD, slope |
| Chest ACC (3-axis) | vector magnitude | mean, SD |
| Chest EMG | 20 Hz high-pass | RMS |
| Wrist BVP (64 Hz) | 0.7–3.5 Hz band-pass, pulse peak detection | HR mean, HR SD, RMSSD |
| Wrist EDA (4 Hz) | 1 Hz low-pass | mean, SD, min, max, slope, # SCR peaks |
| Wrist temperature (4 Hz) | none | mean, SD, slope |
| Wrist ACC (32 Hz, 3-axis) | converted to g, vector magnitude | mean, SD |

Beat-to-beat intervals outside 0.33–1.5 s (40–180 bpm) are discarded as artifacts. A window with fewer than 10 valid beats gets missing (NaN) heart-rate features rather than an unreliable estimate. Missing values are therefore **produced by the pipeline** and reflect signal quality, which is assessed in Section 6.

In [ ]:
WINDOW_S, STEP_S = 60, 30
KEEP_LABELS = {1: "baseline", 2: "stress", 3: "amusement"}
MIN_BEATS = 10
IBI_RANGE = (0.33, 1.5)          # seconds, i.e. 40-180 bpm
SCR_MIN_AMPLITUDE = 0.01         # µS, minimum prominence of a skin conductance response

def sos(kind, cutoff, fs, order=4):
    return signal.butter(order, cutoff, btype=kind, fs=fs, output="sos")

def flat(x):
    return np.asarray(x, dtype=float).ravel()

def slope_per_min(y, fs):
    if len(y) < 2: return np.nan
    t = np.arange(len(y)) / fs / 60
    return np.polyfit(t, y, 1)[0]

def ibi_features(peaks, fs, prefix, include_sdnn=True):
    ibi = np.diff(peaks) / fs
    ibi = ibi[(ibi >= IBI_RANGE[0]) & (ibi <= IBI_RANGE[1])]
    out = {f"{prefix}_hr_mean": np.nan, f"{prefix}_hr_std": np.nan, f"{prefix}_rmssd": np.nan}
    if include_sdnn: out[f"{prefix}_sdnn"] = np.nan
    if len(ibi) >= MIN_BEATS:
        hr = 60 / ibi
        out[f"{prefix}_hr_mean"] = hr.mean()
        out[f"{prefix}_hr_std"] = hr.std()
        out[f"{prefix}_rmssd"] = 1000 * np.sqrt(np.mean(np.diff(ibi) ** 2))
        if include_sdnn: out[f"{prefix}_sdnn"] = 1000 * ibi.std()
    return out

def eda_features(y, fs, prefix):
    pk, _ = signal.find_peaks(y, prominence=SCR_MIN_AMPLITUDE, distance=fs)
    return {f"{prefix}_eda_mean": y.mean(), f"{prefix}_eda_std": y.std(), f"{prefix}_eda_min": y.min(),
            f"{prefix}_eda_max": y.max(), f"{prefix}_eda_slope": slope_per_min(y, fs),
            f"{prefix}_eda_scr_count": len(pk)}

def preprocess_subject(d):
    # Filter each signal once over the whole recording (avoids edge effects in short windows)
    c, w = d["signal"]["chest"], d["signal"]["wrist"]
    p = {}
    # Chest ECG: band-pass, then derivative-squared energy (polarity-independent R-peak emphasis)
    ecg = signal.sosfiltfilt(sos("band", [5, 15], 700), flat(c["ECG"]))
    energy = np.convolve(np.diff(ecg, prepend=ecg[0]) ** 2, np.ones(105) / 105, mode="same")  # 150 ms
    p["ecg_energy"] = energy
    p["chest_eda"] = signal.sosfiltfilt(sos("low", 1.0, 700), flat(c["EDA"]))[::175]          # 4 Hz
    resp10 = signal.sosfiltfilt(sos("low", 5.0, 700), flat(c["Resp"]))[::70]                  # 10 Hz
    p["resp"] = signal.sosfiltfilt(sos("band", [0.1, 0.7], 10, order=2), resp10)
    p["chest_temp"] = signal.sosfiltfilt(sos("low", 1.0, 700), flat(c["Temp"]))[::175]        # 4 Hz
    p["chest_acc"] = np.linalg.norm(np.asarray(c["ACC"], dtype=float), axis=1)
    p["emg"] = signal.sosfiltfilt(sos("high", 20, 700), flat(c["EMG"]))
    p["bvp"] = signal.sosfiltfilt(sos("band", [0.7, 3.5], 64, order=2), flat(w["BVP"]))
    p["wrist_eda"] = signal.sosfiltfilt(sos("low", 1.0, 4, order=2), flat(w["EDA"]))
    p["wrist_temp"] = flat(w["TEMP"])
    p["wrist_acc"] = np.linalg.norm(np.asarray(w["ACC"], dtype=float) / 64, axis=1)
    return p

def window_features(p, t0):
    def sl(x, fs): return x[int(round(t0 * fs)): int(round((t0 + WINDOW_S) * fs))]
    f = {}
    # chest ECG -> heart rate / HRV
    e = sl(p["ecg_energy"], 700)
    pk, _ = signal.find_peaks(e, distance=int(IBI_RANGE[0] * 700), height=0.3 * np.percentile(e, 99))
    f.update(ibi_features(pk, 700, "chest_ecg"))
    f.update(eda_features(sl(p["chest_eda"], 4), 4, "chest"))
    r = sl(p["resp"], 10)
    rpk, _ = signal.find_peaks(r, distance=15, prominence=0.3 * r.std())
    f["chest_resp_rate"] = len(rpk) * 60 / WINDOW_S
    f["chest_resp_std"] = r.std()
    tc = sl(p["chest_temp"], 4)
    f.update({"chest_temp_mean": tc.mean(), "chest_temp_std": tc.std(), "chest_temp_slope": slope_per_min(tc, 4)})
    a = sl(p["chest_acc"], 700)
    f.update({"chest_acc_mean": a.mean(), "chest_acc_std": a.std()})
    f["chest_emg_rms"] = np.sqrt(np.mean(sl(p["emg"], 700) ** 2))
    # wrist BVP -> pulse rate / HRV
    b = sl(p["bvp"], 64)
    bpk, _ = signal.find_peaks(b, distance=int(IBI_RANGE[0] * 64), prominence=0.3 * b.std())
    f.update(ibi_features(bpk, 64, "wrist_bvp", include_sdnn=False))
    f.update(eda_features(sl(p["wrist_eda"], 4), 4, "wrist"))
    tw = sl(p["wrist_temp"], 4)
    f.update({"wrist_temp_mean": tw.mean(), "wrist_temp_std": tw.std(), "wrist_temp_slope": slope_per_min(tw, 4)})
    aw = sl(p["wrist_acc"], 32)
    f.update({"wrist_acc_mean": aw.mean(), "wrist_acc_std": aw.std()})
    return f

The loop below processes one subject at a time (each pickle is a few hundred MB in memory), extracts window features, and records raw-data quality information for Section 6.

In [ ]:
feature_rows, raw_qc, window_qc = [], [], []
t_start = time.time()
for path in subject_files:
    d = load_subject(path)
    sid = d["subject"]
    label = np.asarray(d["label"]).astype(int).ravel()
    c, w = d["signal"]["chest"], d["signal"]["wrist"]

    # raw-data quality record
    dur_chest = len(label) / 700
    dur_wrist = {k: len(w[k]) / FS["wrist"][k] for k in w}
    qc = {"subject": sid, "chest_min": dur_chest / 60, "wrist_min": min(dur_wrist.values()) / 60,
          "chest_minus_wrist_s": dur_chest - min(dur_wrist.values()),
          "label_len_matches_chest": all(len(c[k]) == len(label) for k in c),
          "n_NaN_chest": int(sum(np.isnan(np.asarray(c[k], dtype=float)).sum() for k in c)),
          "n_NaN_wrist": int(sum(np.isnan(np.asarray(w[k], dtype=float)).sum() for k in w))}
    counts = np.bincount(label, minlength=8)
    for k in range(8):
        qc[f"min_label_{k}"] = counts[k] / 700 / 60
    raw_qc.append(qc)

    # windowing
    p = preprocess_subject(d)
    usable = min(dur_chest, min(dur_wrist.values()))
    n_mixed = n_excluded = 0
    for t0 in np.arange(0, usable - WINDOW_S + 1e-9, STEP_S):
        lw = label[int(t0 * 700): int((t0 + WINDOW_S) * 700)]
        u = np.unique(lw)
        if len(u) > 1:
            n_mixed += 1; continue
        if u[0] not in KEEP_LABELS:
            n_excluded += 1; continue
        row = {"subject": sid, "t_start_s": t0, "condition": KEEP_LABELS[u[0]], "stress": int(u[0] == 2)}
        row.update(window_features(p, t0))
        feature_rows.append(row)
    n_kept = sum(r["subject"] == sid for r in feature_rows)
    window_qc.append({"subject": sid, "kept": n_kept, "dropped_mixed_label": n_mixed,
                      "dropped_excluded_label": n_excluded})
    print(f"{sid:>4}: {n_kept:3d} windows kept ({time.time() - t_start:5.0f} s elapsed)")
    del d, p, label, c, w; gc.collect()

df = pd.DataFrame(feature_rows)
raw_qc = pd.DataFrame(raw_qc).set_index("subject")
window_qc = pd.DataFrame(window_qc).set_index("subject")
META_COLS = ["subject", "t_start_s", "condition", "stress"]
FEATURES = [col for col in df.columns if col not in META_COLS]
df.to_csv(OUT_DIR / "wesad_window_features.csv", index=False)
print(f"\nFeature table: {df.shape[0]} windows x {len(FEATURES)} features "
      f"from {df['subject'].nunique()} subjects -> saved to {OUT_DIR / 'wesad_window_features.csv'}")

## 4. Descriptive statistics

### 4.1 Structure of the analysis table
One row per 60-second window. `subject` is the grouping variable, `condition` is the original three-level label, and `stress` is the binary target.

In [ ]:
print(f"Shape: {df.shape}")
print(df.dtypes.value_counts().to_string())
display(df.head())

### 4.2 Target variable

In [ ]:
tgt = (df.groupby(["stress", "condition"]).size().rename("n_windows").reset_index())
tgt["share_%"] = 100 * tgt["n_windows"] / len(df)
display(tgt)
prev = df["stress"].mean()
print(f"Stress prevalence: {prev:.1%}  (non-stress : stress = {(1 - prev) / prev:.1f} : 1)")
print("\nWindows per subject and condition:")
display(pd.crosstab(df["subject"], df["condition"], margins=True))

### 4.3 Predictor variables
Summary statistics for all 32 features (pooled across subjects and classes).

In [ ]:
desc = df[FEATURES].describe().T
desc["skew"] = df[FEATURES].skew()
desc["n_missing"] = df[FEATURES].isna().sum()
display(desc[["count", "n_missing", "mean", "std", "min", "25%", "50%", "75%", "max", "skew"]])

### 4.4 Features by class
Mean and SD for each class, with a standardized mean difference (Cohen's *d*, stress minus non-stress) to rank how strongly each feature separates the classes *when pooled across subjects*. Pooling mixes within-person and between-person differences, so these are descriptive only; the modelling stage will use within-subject normalization and subject-wise validation.

In [ ]:
g = df.groupby("stress")[FEATURES]
m, s, n = g.mean().T, g.std().T, g.count().T
pooled_sd = np.sqrt(((n[0] - 1) * s[0] ** 2 + (n[1] - 1) * s[1] ** 2) / (n[0] + n[1] - 2))
by_class = pd.DataFrame({"non-stress mean": m[0], "non-stress SD": s[0],
                         "stress mean": m[1], "stress SD": s[1],
                         "cohens_d": (m[1] - m[0]) / pooled_sd})
by_class = by_class.reindex(by_class["cohens_d"].abs().sort_values(ascending=False).index)
display(by_class)
top = by_class.index[:3].tolist()
print("Largest pooled separation:", ", ".join(f"{f} (d = {by_class.loc[f, 'cohens_d']:+.2f})" for f in top))

## 5. Exploratory visualizations
(Visualization 1, the raw signal overview, is in Section 2.)

### Visualization 2: class balance by subject

In [ ]:
ct = pd.crosstab(df["subject"], df["condition"])[["baseline", "amusement", "stress"]]
ct = ct.loc[sorted(ct.index, key=lambda s: int(s[1:]))]
fig, ax = plt.subplots(figsize=(10, 3.8))
bottom = np.zeros(len(ct))
for cond, color in [("baseline", COND_COLORS[1]), ("amusement", COND_COLORS[3]), ("stress", COND_COLORS[2])]:
    ax.bar(ct.index, ct[cond], bottom=bottom, color=color, label=cond, edgecolor="white", lw=0.5)
    bottom += ct[cond].values
ax.set_ylabel("60-s windows")
ax.set_title("Labeled 60-second windows per subject and condition")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1))
fig.tight_layout(); fig.savefig(FIG_DIR / "fig2_class_balance.png", bbox_inches="tight"); plt.show()

### Visualization 3: key features, stress vs. non-stress

In [ ]:
KEY = ["chest_eda_mean", "wrist_eda_mean", "chest_ecg_hr_mean", "wrist_bvp_hr_mean",
       "chest_resp_rate", "wrist_temp_slope"]
labels_txt = {0: "non-stress", 1: "stress"}
fig, axes = plt.subplots(2, 3, figsize=(11, 6))
for ax, feat in zip(axes.ravel(), KEY):
    data = [df.loc[df["stress"] == k, feat].dropna() for k in (0, 1)]
    bp = ax.boxplot(data, widths=0.55, patch_artist=True, showfliers=False, medianprops={"color": "black"})
    for patch, color in zip(bp["boxes"], [COND_COLORS[1], COND_COLORS[2]]):
        patch.set_facecolor(color); patch.set_alpha(0.8)
    for i, y in enumerate(data, start=1):     # jittered points
        ax.scatter(np.random.normal(i, 0.06, len(y)), y, s=3, alpha=0.25, color="#333333")
    ax.set_xticks([1, 2], [labels_txt[0], labels_txt[1]])
    ax.set_title(feat, fontsize=10)
fig.suptitle("Distribution of selected features by class (pooled across subjects)")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig3_features_by_class.png", bbox_inches="tight"); plt.show()

### Visualization 4: between-subject vs. within-subject variation
Each line is one participant: their average wrist EDA and chest heart rate in non-stress vs. stress windows. If people differ from each other more than they change under stress, a model trained on raw values will partly learn *who* someone is rather than *how stressed* they are. This plot motivates per-subject normalization.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, feat in zip(axes, ["wrist_eda_mean", "chest_ecg_hr_mean"]):
    subj_means = df.groupby(["subject", "stress"])[feat].mean().unstack()
    for sid, r in subj_means.iterrows():
        up = r[1] > r[0]
        ax.plot([0, 1], [r[0], r[1]], marker="o", ms=4, lw=1,
                color=COND_COLORS[2] if up else "#5b7083", alpha=0.8)
    ax.set_xticks([0, 1], ["non-stress", "stress"]); ax.set_xlim(-0.3, 1.3)
    n_up = int((subj_means[1] > subj_means[0]).sum())
    ax.set_title(f"{feat}\n{n_up} of {len(subj_means)} subjects increase under stress", fontsize=10)
fig.suptitle("Per-subject mean feature values, non-stress vs. stress")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig4_subject_variability.png", bbox_inches="tight"); plt.show()

### Visualization 5: feature correlation matrix
Spearman correlation among the 32 features, to flag redundancy before modelling.

In [ ]:
corr = df[FEATURES].corr(method="spearman")
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURES)), FEATURES, rotation=90, fontsize=7)
ax.set_yticks(range(len(FEATURES)), FEATURES, fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.75, label="Spearman ρ")
ax.set_title("Feature correlations (Spearman)")
fig.tight_layout(); fig.savefig(FIG_DIR / "fig5_feature_correlation.png", bbox_inches="tight"); plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
         .rename("rho").reset_index().rename(columns={"level_0": "feature_1", "level_1": "feature_2"}))
pairs = pairs.reindex(pairs["rho"].abs().sort_values(ascending=False).index)
print("Most strongly correlated feature pairs (|ρ| > 0.8):")
display(pairs[pairs["rho"].abs() > 0.8].head(15))

## 6. Data quality assessment

### 6.1 Raw signals: missing values, label coding, and device alignment

In [ ]:
display(raw_qc[["chest_min", "wrist_min", "chest_minus_wrist_s", "label_len_matches_chest",
                "n_NaN_chest", "n_NaN_wrist"]])
print(f"Raw NaN values across all subjects: chest = {raw_qc['n_NaN_chest'].sum()}, "
      f"wrist = {raw_qc['n_NaN_wrist'].sum()}")
print(f"Chest label vector length matches every chest channel: {raw_qc['label_len_matches_chest'].all()}")
print(f"Largest chest-vs-wrist duration difference: {raw_qc['chest_minus_wrist_s'].abs().max():.1f} s")

In [ ]:
lab_cols = [f"min_label_{k}" for k in range(8)]
lab_min = raw_qc[lab_cols].rename(columns={f"min_label_{k}": LABEL_NAMES[k] for k in range(8)})
total = lab_min.sum(axis=1)
summary = pd.DataFrame({
    "total_min": total,
    "usable_1-3_%": 100 * lab_min[["baseline", "stress", "amusement"]].sum(axis=1) / total,
    "transient_0_%": 100 * lab_min["transient/undefined"] / total,
    "meditation_4_%": 100 * lab_min["meditation"] / total,
    "ignore_5-7_%": 100 * lab_min[["ignore (5)", "ignore (6)", "ignore (7)"]].sum(axis=1) / total})
display(summary)
print(f"Across all subjects, {summary['transient_0_%'].mean() + summary['ignore_5-7_%'].mean():.0f}% "
      "of raw samples are coded 0 or 5-7 (not usable as ground truth).")

### 6.2 Windowing losses

In [ ]:
display(window_qc)
tot = window_qc.sum()
print(f"Kept {tot['kept']} windows; dropped {tot['dropped_mixed_label']} that crossed a condition boundary "
      f"and {tot['dropped_excluded_label']} that were entirely transient, meditation, or ignored.")

### 6.3 Feature-level missingness and implausible values

In [ ]:
miss = df[FEATURES].isna().sum()
miss = miss[miss > 0].sort_values(ascending=False)
if len(miss):
    print("Features with missing values (failed beat/breath detection):")
    display(pd.DataFrame({"n_missing": miss, "%": 100 * miss / len(df)}))
    print("\nMissing wrist pulse-rate by class (%):")
    display(df.groupby("condition")["wrist_bvp_hr_mean"].apply(lambda s: 100 * s.isna().mean()))
else:
    print("No missing feature values.")

checks = {
    "chest ECG HR outside 45-150 bpm": ((df["chest_ecg_hr_mean"] < 45) | (df["chest_ecg_hr_mean"] > 150)).sum(),
    "wrist BVP HR outside 45-150 bpm": ((df["wrist_bvp_hr_mean"] < 45) | (df["wrist_bvp_hr_mean"] > 150)).sum(),
    "chest vs wrist HR differ by > 15 bpm": ((df["chest_ecg_hr_mean"] - df["wrist_bvp_hr_mean"]).abs() > 15).sum(),
    "wrist EDA mean < 0.05 µS (poor skin contact)": (df["wrist_eda_mean"] < 0.05).sum(),
    "wrist temp outside 25-38 °C": ((df["wrist_temp_mean"] < 25) | (df["wrist_temp_mean"] > 38)).sum(),
    "chest temp outside 25-38 °C": ((df["chest_temp_mean"] < 25) | (df["chest_temp_mean"] > 38)).sum(),
}
z = (df[FEATURES] - df[FEATURES].mean()) / df[FEATURES].std()
checks["windows with any |z| > 4"] = int((z.abs() > 4).any(axis=1).sum())
checks["constant (zero-variance) features"] = int((df[FEATURES].std() == 0).sum())
checks["duplicate rows"] = int(df.duplicated().sum())
display(pd.Series(checks, name="n_windows_or_features").to_frame())

### 6.4 Discussion: data quality, anticipated preprocessing, and challenges

**What the checks show.** The dataset authors released the pickles already synchronized and cleaned, so the raw signals are expected to contain no NaNs, and the label vector should align sample-for-sample with the chest channels; Section 6.1 verifies both. The real "improperly coded" observations are in the labels: a substantial share of every recording is coded 0 (transitions between conditions) or 5–7 ("ignore"). These are not usable ground truth and are excluded. Missing values in the feature table are created by the pipeline when beat or breath detection fails. The wrist pulse sensor (BVP) is the most fragile because it is optical and sensitive to motion, so its missingness is broken down by condition in Section 6.3 before any imputation. If pulse detection fails more often during stress (people gesture while giving a speech), the missingness itself is informative and not random.

**Anticipated preprocessing.**
1. *Label handling:* keep labels 1–3, recode to binary, drop mixed windows (done here). Test a sensitivity analysis with meditation added to non-stress.
2. *Signal cleaning:* the filters above are a reasonable first pass; I will visually inspect beat detection on a few windows per subject and tune thresholds, especially for wrist BVP.
3. *Missing values:* impute heart-rate features with the subject's own median (fit inside each training fold) and add a missingness indicator; compare with simply dropping the affected windows.
4. *Per-subject normalization:* z-score each feature within subject (Visualization 4 shows that between-person differences are large). In a real deployment this corresponds to a short personal calibration period, which fits the n-of-1 framing of wearable monitoring.
5. *Redundancy:* several features are highly correlated (e.g., EDA mean/min/max, chest vs. wrist heart rate). Regularized models or feature selection will handle this.
6. *Scaling and class weights:* standardize features for linear models; use class weights because stress is the minority class.

**Potential challenges.**
- **Only 15 people.** About 1,000 windows sounds like enough, but windows from the same person are highly correlated, and overlapping windows share 30 seconds of data. A random train/test split would leak information and inflate performance. Evaluation must be subject-wise (leave-one-subject-out).
- **Confounding of stress with protocol order and context.** In WESAD, stress always involves speaking and standing in a different room. Some "stress" signal (motion, posture, respiration changes from talking) may reflect the task rather than psychological stress. Comparing models with and without accelerometer features will help show how much the model relies on movement.
- **Sampling-rate alignment.** Handled by defining windows in seconds rather than resampling, but heart-rate estimates from the 64 Hz wrist BVP are coarser and noisier than from the 700 Hz chest ECG.
- **Generalizability.** The participants are a small, homogeneous group (mostly male university students) in a controlled lab protocol. Performance will not transfer directly to free-living stress.
- **Window length trade-off.** Longer windows give more stable HRV estimates but fewer observations and coarser timing. 60 s is a starting point to revisit.

## 7. Project planning

**Proposed prediction problem.** Given a 60-second window of physiological signals from wearable sensors, predict whether the person is under acute psychosocial stress. The question I most want to answer: *how much of the chest-sensor performance can a consumer-style wrist device alone achieve?* I will compare three feature sets: chest only, wrist only, and both combined.

**Target variable.** `stress` (binary): 1 = TSST stress condition (WESAD label 2); 0 = non-stress (baseline, label 1, and amusement, label 3). Unit of observation: one 60-second window; grouping variable: `subject`.

**Validation design.** Leave-one-subject-out cross-validation (15 folds). Every preprocessing step that learns from data (imputation, scaling, feature selection) is fit inside the training folds only.

**Candidate models.** Logistic regression with L1/L2 regularization as an interpretable baseline, then random forest and gradient boosting. Time permitting, a 1-D CNN on the raw signals as a comparison with hand-crafted features.

**Candidate evaluation metrics.**
- *Primary:* **macro-averaged F1**, computed per held-out subject and averaged. It weights both classes equally despite the roughly 1 : 2.5 imbalance, and the original WESAD paper reports F1 alongside accuracy for this same binary task (about 93% accuracy using all chest modalities), so my results can be compared with the published benchmark.
- *Secondary:* AUROC (threshold-free ranking quality), balanced accuracy, and sensitivity/specificity for the stress class. In a monitoring application, missed stress episodes and false alarms have different costs, so I'll report both.
- I'll also report the spread of per-subject scores, not just the mean. A model that works well for 12 people and fails for 3 is a different finding from one that is mediocre for everyone.

**Timeline.** Week 4: finalize signal cleaning and features; Week 5: baseline models with LOSO; Week 6: model comparison and chest-vs-wrist analysis; Week 7: interpretation (feature importance, per-subject errors) and write-up.

## 8. Requirements check and session info

In [ ]:
n_obs, n_feat = len(df), len(FEATURES)
target_values = sorted(df["stress"].unique().tolist())
checks = pd.DataFrame([
    ("Binary target", f"stress ∈ {target_values}", target_values == [0, 1]),
    ("≥ 300 observations", f"{n_obs} windows", n_obs >= 300),
    ("≥ 10 predictor variables", f"{n_feat} features", n_feat >= 10),
    ("Subjects", f"{df['subject'].nunique()}", df["subject"].nunique() >= 2),
], columns=["requirement", "value", "met"])
display(checks)
assert checks["met"].all(), "A project requirement is not met"
print(f"All requirements met. Total runtime of feature extraction: {time.time() - t_start:.0f} s")
print(f"Figures saved to {FIG_DIR.resolve()}; feature table saved to {(OUT_DIR / 'wesad_window_features.csv').resolve()}")

### References
- Schmidt P, Reiss A, Duerichen R, Marberger C, Van Laerhoven K. Introducing WESAD, a multimodal dataset for wearable stress and affect detection. *Proceedings of the 20th ACM International Conference on Multimodal Interaction (ICMI)*, 2018:400–408. <https://doi.org/10.1145/3242969.3242985>
- UCI Machine Learning Repository: WESAD. <https://archive.ics.uci.edu/dataset/465/wesad+wearable+stress+and+affect+detection>
- Kirschbaum C, Pirke KM, Hellhammer DH. The 'Trier Social Stress Test': a tool for investigating psychobiological stress responses in a laboratory setting. *Neuropsychobiology*, 1993;28(1–2):76–81.